# 10. Feature Engineering (피처 엔지니어링)

## 목적
전처리된 데이터에서 ML 학습용 파생 피처 생성

## 입력
- `preprocessed.csv`: 전처리된 데이터 (결측 처리 완료)

## 출력
- `features_final.csv`: 최종 피처셋 (35개 피처)

## 최종 피처 구성 (35개)
| 구분 | 피처 | 개수 |
|------|------|------|
| Vitals | hr, rr, spo2, sbp, dbp, mbp, temp | 7 |
| Vitals 통계 | hr_max, rr_max, spo2_min, sbp_min | 4 |
| Labs | creatinine, wbc, platelets, potassium, sodium, lactate | 6 |
| GCS | gcs_eye, gcs_verbal, gcs_motor, gcs_total | 4 |
| Urine | urine_ml_6h, urine_ml_kg_hr_avg, oliguria_flag | 3 |
| 파생 | shock_index, anchor_age, news_score, mews_score | 4 |
| Flags | lactate_missing, abga_checked, is_readmission | 3 |
| Delta | hr_delta, sbp_delta, spo2_delta, lactate_delta, gcs_total_delta | 5 |
| Slope | hr_slope, sbp_slope, spo2_slope, lactate_slope, gcs_total_slope | 5 |

## 피처 엔지니어링 범위
- [x] Slope 피처 (5개)

In [ ]:
import pandas as pd
import numpy as np
import os

INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

print("=== 10. Feature Engineering 시작 ===")

## Step 1: 데이터 로드

In [ ]:
print("Step 1: 데이터 로드")

df = pd.read_csv(os.path.join(INPUT_DIR, 'features_final.csv'))
df = df.sort_values(['stay_id', 'observation_hour']).reset_index(drop=True)

print(f"✓ 데이터 로드 완료: {len(df):,} rows")
print(f"\n고유 환자: {df['stay_id'].nunique():,}명")
print(f"\n고유 stay: {df['stay_id'].nunique():,}개")
print(f"\n컬럼 수: {len(df.columns)}개")

## Step 1: Slope Feature

In [ ]:
print("Step 1: Slope 피처 생성 (6시간 고정 윈도우)")

delta_features = ['hr_delta', 'sbp_delta', 'spo2_delta', 'lactate_delta', 'gcs_total_delta']

for col in delta_features:
    if col not in df.columns:
        print(f"  ⚠️ {col} 없음, 스킵")
        continue

    slope_col = col.replace('_delta', '_slope')

    # 6시간 고정 윈도우 → 시간당 변화율
    df[slope_col] = df[col] / 6.0

    print(f"  ✓ {slope_col}: mean={df[slope_col].mean():.3f}, std={df[slope_col].std():.3f}")

print(f"\n✓ Slope 피처 {len(delta_features)}개 생성 완료")


## Step 6: 최종 피처 정리 및 검증

In [ ]:
print("Step 6: 최종 피처 정리")

# 최종 36개 피처 정의
final_features = {
    'vitals': ['hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp', 'temp'],
    'vitals_stat': ['hr_max', 'rr_max', 'spo2_min', 'sbp_min'],
    'labs': ['creatinine', 'wbc', 'platelets', 'potassium', 'sodium', 'lactate'],
    'gcs': ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total'],
    'urine': ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag'],
    'derived': ['shock_index', 'anchor_age', 'news_score', 'mews_score'],
    'flags': ['lactate_missing', 'abga_checked', 'is_readmission'],
    'delta': ['hr_delta', 'sbp_delta', 'spo2_delta', 'lactate_delta', 'gcs_total_delta'],
    'slope': ['hr_slope', 'sbp_slope', 'spo2_slope', 'lactate_slope', 'gcs_total_slope']
}

# 전체 피처 리스트
all_features = []
for group, cols in final_features.items():
    all_features.extend(cols)

print(f"\n=== 최종 피처 구성 ({len(all_features)}개) ===")
for group, cols in final_features.items():
    existing = [c for c in cols if c in df.columns]
    missing_cols = [c for c in cols if c not in df.columns]
    status = "✓" if len(missing_cols) == 0 else "⚠️"
    print(f"  {status} [{group}] {len(existing)}/{len(cols)}개")
    if missing_cols:
        print(f"      없음: {missing_cols}")

In [ ]:
# 결측 확인
print("\n=== 결측 확인 ===")
existing_features = [c for c in all_features if c in df.columns]
total_missing = df[existing_features].isna().sum().sum()

if total_missing == 0:
    print(f"✓ {len(existing_features)}개 피처 모두 결측 없음")
else:
    print(f"⚠️ 총 결측: {total_missing:,}건")
    missing_detail = df[existing_features].isna().sum()
    print(missing_detail[missing_detail > 0])

In [ ]:
# 피처 기술 통계
print("\n=== 피처 기술 통계 ===")
print(df[existing_features].describe().round(2))

## Step 7: 최종 DataFrame 구성

In [ ]:
print("Step 7: 최종 DataFrame 구성")

# ID 컬럼
id_cols = ['stay_id', 'subject_id', 'hadm_id', 'observation_hour', 'observation_start', 'observation_end']

# 레이블 컬럼
label_cols = [col for col in df.columns if 'next_' in col]

# 최종 컬럼 순서
final_cols = (
    [c for c in id_cols if c in df.columns] +
    [c for c in all_features if c in df.columns] +
    label_cols
)

df_final = df[final_cols].copy()

print(f"\n=== 최종 DataFrame ===")
print(f"  행 수: {len(df_final):,}")
print(f"  컬럼 수: {len(df_final.columns)}")
print(f"    - ID: {len([c for c in id_cols if c in df_final.columns])}개")
print(f"    - 피처: {len([c for c in all_features if c in df_final.columns])}개")
print(f"    - 레이블: {len(label_cols)}개")

## Step 8: 저장

In [ ]:
print("Step 8: 저장")

output_path = os.path.join(OUTPUT_DIR, 'features_final_v2.csv')
df_final.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: features_final.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_final):,}개")
print(f"  - 컬럼 수: {len(df_final.columns)}개")
print(f"  - 경로: {output_path}")

In [ ]:
# 피처 목록 출력
print(f"\n=== 최종 피처 목록 ({len(existing_features)}개) ===")
for group, cols in final_features.items():
    print(f"\n[{group.upper()}]")
    for col in cols:
        if col in df_final.columns:
            print(f"  ✓ {col}")
        else:
            print(f"  ✗ {col} (없음)")

In [ ]:
# 레이블 분포
print(f"\n=== 레이블 분포 ===")
for col in sorted(label_cols):
    pos_rate = df_final[col].mean() * 100
    pos_count = df_final[col].sum()
    print(f"  {col}: {pos_count:,} ({pos_rate:.2f}%)")

In [ ]:
print("\n" + "="*50)
print("=== 10. Feature Engineering 완료 ===")
print("="*50)
print(f"\n최종 피처: {len(existing_features)}개")
print(f"샘플 수: {len(df_final):,}개")
print(f"환자 수: {df_final['stay_id'].nunique():,}명")

---
## +a: 검증

In [ ]:
# 결측 최종 확인
print("=== 결측 최종 확인 ===")
feature_cols_only = [c for c in all_features if c in df_final.columns]
missing = df_final[feature_cols_only].isna().sum()
if missing.sum() == 0:
    print("✓ 모든 피처 결측 없음")
else:
    print("결측 있는 피처:")
    print(missing[missing > 0])

In [ ]:
# 고상관 피처 확인 (다중공선성)
print("\n=== 고상관 피처 쌍 (>0.95) ===")
feature_cols_only = [c for c in all_features if c in df_final.columns]
corr = df_final[feature_cols_only].corr()

high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.95:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

if high_corr:
    for pair in high_corr[:10]:
        print(f"  {pair[0]} vs {pair[1]}: {pair[2]:.3f}")
else:
    print("  없음")

In [ ]:
# Flags 분포 확인
print("\n=== Flags 분포 ===")
for col in ['lactate_missing', 'abga_checked']:
    if col in df_final.columns:
        ones = (df_final[col] == 1).sum()
        zeros = (df_final[col] == 0).sum()
        print(f"  {col}: 1={ones:,} ({ones/len(df_final)*100:.1f}%), 0={zeros:,} ({zeros/len(df_final)*100:.1f}%)")

In [ ]:
# 샘플 데이터 확인
print("\n=== 샘플 데이터 ===")
df_final.head()